# Carvana: earlier intraday experiment (reference)

This preserves the source-to-SQL walkthrough and coverage audits previously inside
notebook 20. These retained morning searches are **not separate daily snapshots**.
Use [notebook 20](20_carvana_history_analysis.ipynb) for the daily inventory and
page-status study. Both notebooks run offline and read-only.

In [ ]:
from pathlib import Path
import json
import sqlite3
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir():
    ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from vehicle_tracker.carvana import parse_capture
from vehicle_tracker.history import read_history, comparison_checks, classify_changes

## Follow a retained request into a comparison

Notebook 10 explains POST requests and pagination. This reference uses the earlier
Tesla Model 3 searches and their repeat from one morning. A missing VIN is an
observed disappearance; an asking-price change is not a transaction-price change.

`read_history` opens the existing SQLite database read-only. Its three tables are
`query_runs` (search attempts), `captures` (saved pages), and `observations`
(listings on those pages). The explicit config selects the before/after reports.

In [ ]:
CONFIG_PATH = ROOT / 'config/carvana_history_example.json'
CONFIG = globals().get('CONFIG_OVERRIDE') or json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
DATABASE = Path(globals().get('DATABASE_OVERRIDE') or ROOT / CONFIG['database'])
history_available = DATABASE.is_file()
query_runs = captures = observations = pd.DataFrame()
previous_observations = current_observations = listing_changes = pd.DataFrame()
comparison_allowed = False
source_available = False
normalized = pd.DataFrame()
if history_available:
    query_runs, captures, observations = read_history(DATABASE)
    history_available = not captures.empty
print('Database:', DATABASE)
if not history_available:
    print('NO DATA: import retained reports explicitly before using this analysis. No collection is run here.')

### Which saved searches are being compared?

The next table shows the requested previous/current reports. The source example then shows the actual filters and pagination from a retained page. A ZIP is shopper context; `LocationBasedPrefiltering`, filters and sort must stay consistent for a before/after comparison.

This is a read of saved JSON, not another search request. Notebook 10 explains how that JSON was collected.

In [ ]:
if history_available:
    selected_reports = pd.DataFrame([
        {'period': period, 'requested_report': str((ROOT / path).resolve())}
        for period, key in [('previous', 'previous_reports'), ('current', 'current_reports')]
        for path in CONFIG[key]])
    display(selected_reports)
    parsed_captures = captures[captures.status.eq('parsed')]
    source_available = not parsed_captures.empty and Path(parsed_captures.iloc[0].source_path).is_file()
    if source_available:
        example_capture = parsed_captures.iloc[0]
        source_path = Path(example_capture.source_path)
        source = json.loads(source_path.read_text(encoding='utf-8'))
        print('Source:', source_path, 'Observed:', source['captured_at_utc'])
        display(pd.json_normalize(source['request']))
        display(pd.DataFrame([source['pagination']]))
        display(pd.DataFrame([{'requested_zip': source['requested_zip'], 'returned_zip': source['zip_code'], 'source_sha256': example_capture.capture_id}]))
    else:
        print('NO RETAINED SOURCE: the example source is absent or no page parsed. Stored rows remain exploratory; the source trace and comparison are withheld.')

## 2. Follow one saved vehicle into the database

`parse_capture(source)` turns the saved page into a DataFrame, using the same field rules as notebook 10. The SQL query then retrieves one matching database observation by **capture ID and listing ID**. Matching the capture matters: the same listing can have several observations at different times.

Read the SQL as: "Get this listing's observation and attach the path of the page it came from." `?` placeholders supply the chosen identifiers. `mode=ro` opens SQLite read-only. The VIN check confirms that this example refers to the same vehicle; it is not a full audit of every stored value.

| Source field | Analysis column | Meaning |
| --- | --- | --- |
| `vehicleId` / `vin` | `listing_id` / `vin` | Listing identity / vehicle identity |
| `mileage` | `mileage_miles` | Published odometer in miles |
| `price.total` | `asking_price_usd` | Asking price in USD |
| `isPurchasePending` | `purchase_pending` | Native pending flag, not a sale |
| `vehicleLockType` | `vehicle_lock_type` | Native code; no invented sale meaning |

In [ ]:
if source_available:
    normalized = parse_capture(source)
    display(normalized[['listing_id', 'vin', 'year', 'make', 'model', 'mileage_miles', 'asking_price_usd']].head(8))


if not normalized.empty:
    example_id = normalized.iloc[0].listing_id
    connection = sqlite3.connect(DATABASE.resolve().as_uri() + '?mode=ro', uri=True)
    try:
        stored_example = pd.read_sql_query(
            'SELECT o.*, c.source_path FROM observations o JOIN captures c USING(capture_id) '
            'WHERE o.capture_id=? AND o.listing_id=?', connection,
            params=[example_capture.capture_id, example_id])
    finally:
        connection.close()
    display(stored_example[['listing_id', 'vin', 'observed_at_utc', 'asking_price_usd',
                            'purchase_pending', 'source_path']])
    assert stored_example.iloc[0].vin == normalized.iloc[0].vin
    print('The parsed source and stored observation identify the same vehicle for this capture.')

## 3. Can we fairly compare these collections?

Before subtracting prices or counting missing listings, we must establish that both sides cover the same search population. `comparison_checks` returns a table of rules and pass/fail results:

- Every explicitly selected search exists and completed.
- Filters, ZIP, location setting and sort match, with no missing partitions.
- Actual collection windows are ordered and do not overlap. Reusing an old capture is not a fresh collection.

The cell also displays missing fields and duplicate listing/VIN identities. Unknown prices stay missing; ambiguous identities block the comparison. `comparison_allowed` is the final switch used by sections 4 and 5. If it is false, their change tables are withheld; the separate observed rows can still be inspected.

In [ ]:
if history_available:
    report_to_run = query_runs.set_index('report_path').run_id.to_dict()
    previous_ids = [report_to_run.get(str((ROOT / p).resolve()), 'missing:' + p) for p in CONFIG['previous_reports']]
    current_ids = [report_to_run.get(str((ROOT / p).resolve()), 'missing:' + p) for p in CONFIG['current_reports']]
    coverage_checks = comparison_checks(query_runs, previous_ids, current_ids)
    coverage_checks.loc[len(coverage_checks)] = ['source_trace', source_available, 'OK' if source_available else 'Retained example unavailable; source trace cannot be checked']
    display(coverage_checks)
    selected_runs = query_runs[query_runs.run_id.isin(previous_ids + current_ids)].copy()
    selected_runs['reconciliation_difference'] = selected_runs.stored_rows - selected_runs.reported_total
    display(selected_runs[['report_path', 'query_complete', 'coverage_reason', 'reported_total',
                           'stored_rows', 'reconciliation_difference', 'observation_start', 'observation_end']])
    previous_observations = observations[observations.run_id.isin(previous_ids)].copy()
    current_observations = observations[observations.run_id.isin(current_ids)].copy()
    for label, frame in [('previous', previous_observations), ('current', current_observations)]:
        print(label, 'rows:', len(frame), 'distinct listings:', frame.listing_id.nunique(), 'distinct VINs:', frame.vin.nunique())
        display(frame[['vin', 'year', 'mileage_miles', 'asking_price_usd', 'purchase_pending']].isna().sum().rename('missing').to_frame())
        display(frame[frame.duplicated(['retailer', 'listing_id'], keep=False)])
        display(frame[frame.vin.isna() | frame.duplicated(['retailer', 'vin'], keep=False)])
    comparison_allowed = bool(coverage_checks.passed.all())
    if any(frame.duplicated(['retailer', 'listing_id']).any() or frame.duplicated(['retailer', 'vin']).any() or frame.vin.isna().any() for frame in [previous_observations, current_observations]):
        comparison_allowed = False
        print('BLOCKED: overlapping identities require explicit scope selection.')
    if not comparison_allowed:
        print('BLOCKED: explore the observed rows separately; no absence classification is produced.')

## 4. Line up the same listings and calculate changes

`merge(..., how="outer")` puts the previous and current observations side by side using `(retailer, listing_id)`. "Outer" means keep listings from either collection, including those without a match.

| Join result | What it means here |
| --- | --- |
| `both` | Listing observed in both collections |
| `left_only` | Observed before, not observed later |
| `right_only` | Observed later, not in the previous collection |

For matched rows with the same VIN, the price calculation is simply **current asking price - previous asking price**. Missing prices or conflicting VINs produce no price-change estimate. Native pending/lock changes also require known comparable values.

`classify_changes` adds readable labels and checks earlier retained observations to distinguish a reappearance from a first observation in this history. First observed does not mean newly listed; not observed later does not mean sold. The cell checks the visible subtraction against the helper's result, then shows changed/unmatched rows and their source paths.

In [ ]:
if comparison_allowed:
    joined = previous_observations.merge(current_observations, on=['retailer', 'listing_id'], how='outer',
                                        suffixes=('_before', '_after'), indicator=True, validate='one_to_one')
    display(joined[['listing_id', 'vin_before', 'vin_after', '_merge']].head(12))
    same_vehicle = joined.vin_before.notna() & joined.vin_before.eq(joined.vin_after)
    visible_price_difference = (joined.asking_price_usd_after - joined.asking_price_usd_before).where(
        joined['_merge'].eq('both') & same_vehicle)
    contexts = set(selected_runs.context_json)
    earlier_ids = query_runs.loc[query_runs.context_json.isin(contexts) &
        query_runs.observation_end.lt(query_runs.loc[query_runs.run_id.isin(previous_ids), 'observation_start'].min()), 'run_id']
    seen_before = set(observations.loc[observations.run_id.isin(earlier_ids), 'listing_id'])
    listing_changes = classify_changes(previous_observations, current_observations, seen_before=seen_before)
    pd.testing.assert_series_equal(visible_price_difference, listing_changes.asking_price_change_usd, check_names=False)
    unusual_rows = listing_changes[listing_changes.observation_change.ne('observed_both') |
        listing_changes.asking_price_change_usd.fillna(0).ne(0) | listing_changes.native_status_changed.fillna(False)].copy()
    source_lookup = captures.set_index('capture_id').source_path
    for suffix in ['before', 'after']:
        unusual_rows['source_path_' + suffix] = unusual_rows['capture_id_' + suffix].map(source_lookup)
    display(unusual_rows[['listing_id', 'observation_change', 'asking_price_change_usd', 'native_status_changed',
                          'source_path_before', 'source_path_after']].head(20))

## 5. Read the result

The first table counts the change labels from section 4. The second describes the selected current cohort by make/model/year, including missing prices. Counts and medians refer to this cohort only. A zero price change is an observed unchanged price; a missing change is unknown or incomparable.

The printed totals separate asking-price changes from native-status changes. None is an estimated-sales count. For an individual vehicle, the next lookup traces both collections back to their source pages.

In [ ]:
if comparison_allowed:
    change_summary = listing_changes.groupby('observation_change', dropna=False).agg(listings=('listing_id', 'nunique'))
    current_inventory_mix = current_observations.groupby(['make', 'model', 'year'], dropna=False).agg(
        listings=('listing_id', 'nunique'), median_asking_price_usd=('asking_price_usd', 'median'),
        missing_prices=('asking_price_usd', lambda values: values.isna().sum()))
    display(change_summary)
    display(current_inventory_mix)
    known_changes = listing_changes.asking_price_change_usd.dropna()
    print('Matched nonzero asking-price changes:', int(known_changes.ne(0).sum()))
    print('Known native status changes:', int(listing_changes.native_status_changed.fillna(False).sum()))
    print('These are changes between observation windows, not a daily sales estimate.')

### Inspect one vehicle behind the result

`AUDIT_VIN` defaults to the first current VIN; use `AUDIT_VIN_OVERRIDE` to choose another. The lookup uses only the explicitly selected previous/current searches. It attaches source paths, report paths and parser versions to the observations so you can inspect the evidence behind a price or status change.

In [ ]:
if comparison_allowed and not current_observations.empty:
    AUDIT_VIN = globals().get('AUDIT_VIN_OVERRIDE', current_observations.vin.iloc[0])
    selected_vin = observations[observations.vin.eq(AUDIT_VIN) & observations.run_id.isin(previous_ids + current_ids)]
    lineage = selected_vin.merge(captures[['capture_id', 'source_path']], on='capture_id', validate='many_to_one')
    lineage = lineage.merge(query_runs[['run_id', 'report_path', 'context_json', 'normalizer_sha256', 'original_normalizer_sha256']], on='run_id', validate='many_to_one')
    display(lineage[['vin', 'listing_id', 'observed_at_utc', 'asking_price_usd', 'purchase_pending', 'source_path', 'report_path', 'normalizer_sha256', 'original_normalizer_sha256']])
    print('Native-status changes use known evidence; missing values never become completed sales.')
else:
    print('VIN trace unavailable: select a complete comparable intraday cohort.')

## Appendix A. Broader sample and collection diagnostics

These tables answer a different question: "What did the wider trial observe?" They do not change the Tesla before/after comparison above. Partial queries contribute only admitted rows. The union is a historical sample, not complete inventory. Duplicate memberships remain visible, and ZIP prices are not silently collapsed into a newest value.

The collection reports show request budgets, sample-target stops and query outcomes; the following table summarizes explicitly selected wider observations by make and year.

In [ ]:
if history_available:
    for relative in CONFIG.get('collection_reports', []):
        path = ROOT / relative
        if path.exists():
            plan = json.loads(path.read_text(encoding='utf-8'))
            print('Collection plan:', path)
            display(pd.DataFrame([{key: plan.get(key) for key in ['started_utc', 'ended_utc', 'observation_started_utc', 'observation_ended_utc', 'requests', 'target_listings', 'target_reached']}]))
            display(pd.DataFrame(plan['outcomes']).reindex(columns=['query_id', 'status', 'query_complete', 'reason']))
if history_available and CONFIG.get('exploratory_reports'):
    exploratory_ids = [report_to_run.get(str((ROOT / path).resolve())) for path in CONFIG['exploratory_reports']]
    exploratory_runs = query_runs[query_runs.run_id.isin(exploratory_ids)].copy()
    exploratory_runs['reconciliation_difference'] = exploratory_runs.stored_rows - exploratory_runs.reported_total
    exploratory_observations = observations[observations.run_id.isin(exploratory_ids)]
    display(exploratory_runs[['report_path', 'query_complete', 'coverage_reason', 'reported_total', 'stored_rows', 'reconciliation_difference', 'observation_start', 'observation_end']])
    print('Saved sample:', len(exploratory_observations), 'rows;', exploratory_observations.listing_id.nunique(),
          'distinct listing IDs;', exploratory_observations.vin.nunique(), 'distinct VINs')
    broader_mix = exploratory_observations.groupby(['make', 'year'], dropna=False).agg(
        observed_listings=('listing_id', 'nunique'), median_asking_price_usd=('asking_price_usd', 'median'),
        missing_prices=('asking_price_usd', lambda values: values.isna().sum()))
    display(broader_mix)
    display(exploratory_observations[exploratory_observations.duplicated(['retailer', 'listing_id'], keep=False)])
    print('Counts describe the saved sample. Inspect partial and unattempted queries in the collection plan above.')

## Appendix B. Check year partitions and nearby ZIP overlap

The explicit audit reports use Tesla Model 3, years 2024–2025, the same sort and location-feature setting. Verify the displayed contexts before interpreting the identity union. Child years partition the parent. ZIP membership is unioned, never summed; conflicting VIN mappings remain visible.

In [ ]:
if history_available and CONFIG.get('audit_reports'):
    audit_ids = {name: report_to_run.get(str((ROOT / path).resolve())) for name, path in CONFIG['audit_reports'].items()}
    audit_runs = query_runs[query_runs.run_id.isin(audit_ids.values())]
    display(audit_runs[['report_path', 'context_json', 'query_complete', 'observation_start', 'observation_end']])
    audit_ready = len(audit_runs) == len(audit_ids) and audit_runs.query_complete.eq(1).all()
    audit_context_checks = pd.DataFrame()
    if audit_ready:
        contexts = {name: json.loads(audit_runs.set_index('run_id').loc[run_id, 'context_json']) for name, run_id in audit_ids.items()}
        parent_context = contexts['parent']
        expected_filters = {'makes': [{'name': 'Tesla', 'parentModels': [{'name': 'Model 3'}]}], 'year': {'min': 2024, 'max': 2025}}
        expected_contexts = {'parent': dict(parent_context, filters=expected_filters, zip_code='08542'),
                             'nearby_zip': dict(parent_context, filters=expected_filters, zip_code='08540')}
        for year in [2024, 2025]:
            expected_contexts['child_' + str(year)] = dict(expected_contexts['parent'], filters=dict(expected_filters, year={'min': year, 'max': year}))
        audit_context_checks = pd.DataFrame([{'query': name, 'passed': contexts[name] == expected, 'reason': 'Require the declared Tesla years/ZIPs and identical endpoint, sort and feature setting'} for name, expected in expected_contexts.items()])
        display(audit_context_checks)
        audit_ready = bool(audit_context_checks.passed.all())
    if audit_ready:
        identity_sets = {name: set(zip(frame.retailer, frame.listing_id)) for name, run_id in audit_ids.items()
                         for frame in [observations[observations.run_id.eq(run_id)]]}
        child_union = identity_sets['child_2024'] | identity_sets['child_2025']
        zip_union = identity_sets['parent'] | identity_sets['nearby_zip']
        zip_overlap = identity_sets['parent'] & identity_sets['nearby_zip']
        audit_summary = pd.DataFrame([{'parent_listings': len(identity_sets['parent']), 'child_union': len(child_union),
            'missing_from_children': len(identity_sets['parent'] - child_union), 'extra_in_children': len(child_union - identity_sets['parent']),
            'zip_intersection': len(zip_overlap), 'zip_union': len(zip_union), 'incremental_nearby_zip': len(identity_sets['nearby_zip'] - identity_sets['parent'])}])
        display(audit_summary)
        audit_observations = observations[observations.run_id.isin(audit_ids.values())]
        identity_conflicts = audit_observations.groupby(['retailer', 'listing_id']).vin.nunique(dropna=False)
        display(identity_conflicts[identity_conflicts.gt(1)].rename('conflicting_vins').to_frame())
        print('Parent/child identity reconciliation:', identity_sets['parent'] == child_union and not identity_conflicts.gt(1).any())
        print('Matched query contexts; ZIP prices stay separate and national coverage is unverified.')
    else:
        print('BLOCKED: ZIP/partition evidence is unavailable, incomplete or has mismatched context. No complete-union claim.')

## Appendix C. See why an incomplete query blocks comparison

The next cell changes one coverage flag **in memory only**. It demonstrates why an
incomplete query cannot support an absence comparison. The database and real results
remain unchanged. Identical counts do not override a failed coverage check.

In [ ]:
if history_available and comparison_allowed:
    incomplete_example = query_runs.copy(deep=True)
    incomplete_example.loc[incomplete_example.run_id.eq(current_ids[0]), 'query_complete'] = 0
    example_checks = comparison_checks(incomplete_example, previous_ids, current_ids)
    display(example_checks)
    assert not example_checks.passed.all()
    print('BLOCKED EXAMPLE: absence counts are withheld because a requested query is incomplete.')